In [ ]:
import numpy as np
import random
import string
import os
import re


In [ ]:
# Generating random cipher
alphabet = list(string.ascii_lowercase)

def cipher():
    cipher_dict = {}
    shuffled_alphabet = alphabet.copy()
    random.shuffle(shuffled_alphabet)
    cipher_dict = dict(zip(alphabet, shuffled_alphabet))
    return cipher_dict

In [ ]:
# Reading text file
md = []
start = "^"
file_path = os.path.join(os.getcwd(), "Moby_Dick.txt")
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.lower().translate(str.maketrans('', '', string.punctuation)).strip()
        if line:
            for word in line.split():
                if word.isalpha():
                    md.append(f"{start}{word}")

In [ ]:
# Markov model based on char
prob_dict = {}
e = 0.3
unseen = 1e-6 
for word in md:
    n = len(word)
    for i in range(n-1):
        c1 = word[i]
        c2 = word[i+1]
        if c1 not in prob_dict:
            prob_dict[c1] = {}
        prob_dict[c1][c2] = prob_dict[c1].get(c2, e) + 1
for c1,c2s in prob_dict.items():
    total = sum(c2s.values())
    for c2, counts in prob_dict[c1].items():
        prob_dict[c1][c2] = counts/total
# finding probability based on markov mopdel
def likelyhood(message):
    tem_str = []
    char_prob = {}
    prob = 0
    for word in message.split():
        tem_str.append(f"{start}{word}")
    for word in tem_str:
        n = len(word)
        for i in range(n-1):
            c1 = word[i]
            c2 = word[i+1]
            if c1 in prob_dict and c2 in prob_dict[c1]:
                transition_prob = prob_dict[c1][c2]
            else:
                transition_prob = unseen
            p = np.log(transition_prob)
            prob += p
    return prob


In [ ]:
# Encoding
def encode(message, map):
    encoded_words = []
    for word in message.split():
        encoded_word = "".join([map[char] for char in word])
        encoded_words.append(encoded_word)
    return " ".join(encoded_words)
# Decoding
def decode(message, map):
    decoded_words = []
    for word in message.split():
        decoded_word = "".join([map[char] for char in word])
        decoded_words.append(decoded_word)
    return " ".join(decoded_words)


In [ ]:
# Genetic algorithm, finding the best cipher
def evaluation(encrypted_message, map):
    guessed_message = decode(encrypted_message, map)
    probability = likelyhood(guessed_message) 
    return probability

def crossover(cipher1, cipher2):
    p1 = [cipher1[ch] for ch in alphabet]
    p2 = [cipher2[ch] for ch in alphabet]
    child = [None] * 26
    child[:13] = p1[:13]
    used = set(child[:13])
    fill_index = 13
    for letter in p2:
        if letter not in used:
            child[fill_index] = letter
            used.add(letter)
            fill_index += 1
    return {alphabet[i]: child[i] for i in range(26)}

def mutation(map):
    new_map = map.copy()
    a, b = random.sample(alphabet, 2)
    new_map[a], new_map[b] = new_map[b], new_map[a]
    return new_map

def genetic(encrypted_message):
    ciphers = []
    parents = []
    population = 300
    ratio = 0.3
    for i in range(population):
        ciphers.append(cipher())
    for item in ciphers:
        score = evaluation(encrypted_message, item)
        parents.append((score, item))
    prev_best = None
    stagnation = 0
    stagnation_limit = 200
    generation = 0
    max_generations = 2000
    while generation < max_generations:
        parents.sort(key=lambda x: x[0], reverse=True)
        number = int(len(parents) * ratio)
        parents = parents[:number]
        child_temp = []
        while len(child_temp) < population - number:
            parent1, parent2 = random.sample(parents, 2)
            child = crossover(parent1[1], parent2[1])
            child_temp.append(child)
            
        for i in range(int(len(child_temp)*ratio)):
            idx = random.randrange(len(child_temp)) 
            child_temp[idx] = mutation(child_temp[idx])
            if random.random() < 0.5:
                idx = random.randrange(len(child_temp)) 
                child_temp[idx] = mutation(child_temp[idx])
        for c in child_temp:
            scores = evaluation(encrypted_message, c)
            parents.append((scores, c))
        best_score = max(score for score, _ in parents)
        best_cipher = max(parents, key=lambda x: x[0])[1]
        if prev_best is None or best_score > prev_best:
            stagnation = 0  
        else:
            stagnation += 1
        if stagnation >= stagnation_limit:
            break
        prev_best = best_score
        generation += 1
    return decode(encrypted_message, best_cipher), best_cipher

In [ ]:
# Comparing the real message with the model guess
message = "Call me Ishmael. Some years ago never mind how long precisely having little or no money in my purse, and nothing particular to interest me on shore, I thought I would sail about a little and see the watery part of the world. It is a way I have of driving off the spleen and regulating the circulation. Whenever I find myself growing grim about the mouth; whenever it is a damp, drizzly November in my soul; whenever I find myself involuntarily pausing before coffin warehouses, and bringing up the rear of every funeral I meet; and especially whenever my hypos get such an upper hand of me, that it requires a strong moral principle to prevent me from deliberately stepping into the street, and methodically knocking people’s hats off then, I account it high time to get to sea as soon as I can"
message = re.sub(r'[^a-zA-Z ]', '', message).lower()
true_map = cipher()
encrypted = encode(message, true_map)
decrypted ,founded_cipher = genetic(encrypted) 
true_decryption_map = {value: key for key, value in true_map.items()}
true_decryption_map = dict(sorted((v, k) for k, v in true_map.items()))

print("Original:", message)
print("Decrypted:", decrypted)
print(true_decryption_map)
print(founded_cipher)
